# Titanic: какие характеристики пассажиров связаны с выживанием

**Автор:** _имя_ · **Модуль:** Python, итоговая практика · **Данные:** Kaggle *Titanic — Machine Learning from Disaster*, `train.csv` (891 пассажир)

**Вопрос проекта:** какие характеристики пассажиров связаны с выживаемостью.
Данные показывают **связи**, но не доказывают **причины** — все выводы сформулированы как наблюдения.

**Содержание**
1. Аудит данных
2. Очистка
3. Новые признаки
4. Исследовательский анализ (10 вопросов)
5. NumPy-часть
6. Итоговая визуализация и таблица результатов
7. Выводы, ограничения, идеи продолжения
8. Дополнительно: rule-based прогноз

## 0. Настройка

Импорты, seed, относительные пути, единая палитра.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

RAW_PATH = Path("data/raw/train.csv")
PROCESSED_DIR = Path("data/processed")
FIG_DIR = Path("reports/figures")
TABLE_DIR = Path("reports/tables")
for d in (PROCESSED_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Okabe–Ito: различима при дальтонизме
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
SURVIVED_C, DIED_C, MUTED = "#009E73", "#D55E00", "#B0B0B0"
plt.rcParams.update({
    "figure.dpi": 110, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "axes.spines.top": False, "axes.spines.right": False,
    "text.parse_math": False,
})
pd.set_option("display.precision", 3)

## 1. Аудит данных

Размер, типы, data dictionary, группы признаков, пропуски, дубликаты, подозрительные значения, уникальные значения, базовые статистики.

In [3]:
# блок 1
df_raw = pd.read_csv(RAW_PATH)
n_rows, n_cols = df_raw.shape
print(f"{n_rows} строк, {n_cols} столбцов")
print(list(df_raw.columns))


891 строк, 12 столбцов
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [19]:
df_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [20]:
df_raw.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.283,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.100,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S


### 1.1 Гипотезы до очистки

_Что ожидаю увидеть, до расчётов:_

1. ...
2. ...
3. ...

## 2. Очистка

Стратегия для `Age`, `Cabin`, `Embarked`; аномалии `Fare`; единый формат строк; согласованность `SibSp`, `Parch`, `Ticket`, `Cabin`.
Исходный DataFrame не меняется — очистка работает на копии. В конце — журнал преобразований и повторный аудит.

In [21]:
# блок 2
data_dictionary = pd.DataFrame(
    [
        ("PassengerId", "номер строки в файле", "идентификатор"),
        ("Survived", "выжил: 1 — да, 0 — нет (целевая переменная)", "категориальный"),
        ("Pclass", "класс билета: 1 — высший, 3 — низший", "порядковый"),
        ("Name", "фамилия, титул и имя пассажира", "идентификатор"),
        ("Sex", "пол: male / female", "категориальный"),
        ("Age", "возраст в годах (у младенцев дробный)", "числовой"),
        ("SibSp", "число братьев/сестёр и супругов на борту", "числовой"),
        ("Parch", "число родителей и детей на борту", "числовой"),
        ("Ticket", "номер билета (у части пассажиров общий)", "идентификатор"),
        ("Fare", "стоимость билета в фунтах", "числовой"),
        ("Cabin", "номер каюты; первая буква — палуба", "категориальный"),
        ("Embarked", "порт: C — Шербур, Q — Квинстаун, S — Саутгемптон", "категориальный"),
    ],
    columns=["признак", "что означает", "группа"],
)
assert list(data_dictionary["признак"]) == list(df_raw.columns), "словарь не совпадает со столбцами"
data_dictionary["тип в pandas"] = data_dictionary["признак"].map(df_raw.dtypes.astype(str))
data_dictionary


,признак,что означает,группа,тип в pandas
0,PassengerId,номер строки в файле,идентификатор,int64
1,Survived,"выжил: 1 — да, 0 — нет (целевая переменная)",категориальный,int64
2,Pclass,"класс билета: 1 — высший, 3 — низший",порядковый,int64
3,Name,"фамилия, титул и имя пассажира",идентификатор,str
4,Sex,пол: male / female,категориальный,str
5,Age,возраст в годах (у младенцев дробный),числовой,float64
6,SibSp,число братьев/сестёр и супругов на борту,числовой,int64
7,Parch,число родителей и детей на борту,числовой,int64
8,Ticket,номер билета (у части пассажиров общий),идентификатор,str
9,Fare,стоимость билета в фунтах,числовой,float64


In [22]:
FEATURE_GROUPS = data_dictionary.groupby("группа")["признак"].agg(list).to_dict()
for group, cols in FEATURE_GROUPS.items():
    print(f"{group:<15} ({len(cols)}): {', '.join(cols)}")


идентификатор   (3): PassengerId, Name, Ticket
категориальный  (4): Survived, Sex, Cabin, Embarked
порядковый      (1): Pclass
числовой        (4): Age, SibSp, Parch, Fare


### 2.1 Вывод по очистке

...

## 3. Новые признаки

`FamilySize`, `IsAlone`, `Title`, `Deck`, `TicketGroupSize` + дополнительные. Для каждого — какую информацию он добавляет.

In [23]:
# блок 3
missing = pd.DataFrame({
    "пропусков": df_raw.isna().sum(),
    "доля, %": df_raw.isna().mean() * 100,
})
missing[missing["пропусков"] > 0].sort_values("пропусков", ascending=False)


,пропусков,"доля, %"
Cabin,687,77.104
Age,177,19.865
Embarked,2,0.224


In [24]:
print(f"полных дубликатов строк: {df_raw.duplicated().sum()}")
print(f"дубликатов без учёта PassengerId: {df_raw.drop(columns='PassengerId').duplicated().sum()}")


полных дубликатов строк: 0
дубликатов без учёта PassengerId: 0


In [25]:
unique_counts = pd.DataFrame({"уникальных": df_raw.nunique()})
unique_counts["доля от строк, %"] = unique_counts["уникальных"] / len(df_raw) * 100
unique_counts["группа"] = data_dictionary.set_index("признак")["группа"]
unique_counts


,уникальных,"доля от строк, %",группа
PassengerId,891,100.000,идентификатор
Survived,2,0.224,категориальный
Pclass,3,0.337,порядковый
Name,891,100.000,идентификатор
Sex,2,0.224,категориальный
Age,88,9.877,числовой
SibSp,7,0.786,числовой
Parch,7,0.786,числовой
Ticket,681,76.431,идентификатор
Fare,248,27.834,числовой


### 3.1 Что добавляет каждый признак

| признак | как получен | что добавляет |
|---|---|---|
| ... | ... | ... |

## 4. Исследовательский анализ

Для каждого вопроса: расчёт → график → вывод. **Факт** — то, что видно в данных; **предположение** — возможное объяснение, которое данные не доказывают.

### 4.1 Какова общая доля выживших?

In [26]:
# вопрос 4.1
df_raw[FEATURE_GROUPS["числовой"]].describe().T


,count,mean,std,min,25%,50%,75%,max
Age,714.0,29.699,14.526,0.42,20.125,28.000,38.0,80.000
SibSp,891.0,0.523,1.103,0.00,0.000,0.000,1.0,8.000
Parch,891.0,0.382,0.806,0.00,0.000,0.000,0.0,6.000
Fare,891.0,32.204,49.693,0.00,7.910,14.454,31.0,512.329


In [27]:
cat_cols = ["Survived", "Pclass", "Sex", "Embarked"]
pd.concat(
    {col: df_raw[col].value_counts(dropna=False, normalize=True) * 100 for col in cat_cols}
).rename("доля, %")


Survived  0         61.616
          1         38.384
Pclass    3         55.107
          1         24.242
          2         20.651
Sex       male      64.759
          female    35.241
Embarked  S         72.278
          C         18.855
          Q          8.642
          NaN        0.224
Name: доля, %, dtype: float64

In [28]:
q1, q3 = df_raw["Fare"].quantile([0.25, 0.75])
fare_upper = q3 + 1.5 * (q3 - q1)
print(f"верхняя граница IQR для Fare: {fare_upper:.2f}")

pd.Series(
    {
        "Fare == 0": (df_raw["Fare"] == 0).sum(),
        "Fare выше границы IQR": (df_raw["Fare"] > fare_upper).sum(),
        "1-й класс и Fare < 20": ((df_raw["Pclass"] == 1) & (df_raw["Fare"] < 20)).sum(),
        "3-й класс и Fare > 50": ((df_raw["Pclass"] == 3) & (df_raw["Fare"] > 50)).sum(),
        "Age < 1": (df_raw["Age"] < 1).sum(),
        "Age дробный, но >= 1": ((df_raw["Age"] % 1 > 0) & (df_raw["Age"] >= 1)).sum(),
        "Age >= 70": (df_raw["Age"] >= 70).sum(),
        "SibSp >= 5": (df_raw["SibSp"] >= 5).sum(),
        "Parch >= 4": (df_raw["Parch"] >= 4).sum(),
        "Cabin: несколько кают в ячейке": df_raw["Cabin"].str.contains(" ").sum(),
    },
    name="пассажиров",
)


верхняя граница IQR для Fare: 65.63


Fare == 0                          15
Fare выше границы IQR             116
1-й класс и Fare < 20               6
3-й класс и Fare > 50              14
Age < 1                             7
Age дробный, но >= 1               18
Age >= 70                           7
SibSp >= 5                         12
Parch >= 4                         10
Cabin: несколько кают в ячейке     24
Name: пассажиров, dtype: int64

In [29]:
text_cols = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]
has_edge_space = {col: df_raw[col].str.contains(r"^\s|\s$") for col in text_cols}
print(pd.Series({col: mask.sum() for col, mask in has_edge_space.items()}, name="пробелы по краям"))
print(df_raw.loc[has_edge_space["Name"], "Name"].tolist())


Name        2
Sex         0
Ticket      0
Cabin       0
Embarked    0
Name: пробелы по краям, dtype: int64
['Hewlett, Mrs. (Mary D Kingcome) ', 'Daly, Mr. Peter Denis ']


In [30]:
df_raw.loc[df_raw["Fare"] == 0, ["Pclass", "Ticket", "Cabin", "Embarked", "Survived"]]


,Pclass,Ticket,Cabin,Embarked,Survived
179,3,LINE,NaN,S,0
263,1,112059,B94,S,0
271,3,LINE,NaN,S,1
277,2,239853,NaN,S,0
302,3,LINE,NaN,S,0
413,2,239853,NaN,S,0
466,2,239853,NaN,S,0
481,2,239854,NaN,S,0
597,3,LINE,NaN,S,0
633,1,112052,NaN,S,0


**Факт:** ...

**Предположение:** ...

### 4.2 Как выживаемость связана с полом и классом билета?

In [7]:
# вопрос 4.2

**Факт:** ...

**Предположение:** ...

### 4.3 Что меняется при совместном рассмотрении пола и класса?

In [8]:
# вопрос 4.3

**Факт:** ...

**Предположение:** ...

### 4.4 Как с выживаемостью связан возраст?

In [9]:
# вопрос 4.4

**Факт:** ...

**Предположение:** ...

### 4.5 Как с выживаемостью связан размер семьи?

In [10]:
# вопрос 4.5

**Факт:** ...

**Предположение:** ...

### 4.6 Отличается ли выживаемость одиночных пассажиров?

In [11]:
# вопрос 4.6

**Факт:** ...

**Предположение:** ...

### 4.7 Какие различия видны между портами посадки?

In [12]:
# вопрос 4.7

**Факт:** ...

**Предположение:** ...

### 4.8 Связаны ли стоимость билета и палуба с выживаемостью?

In [13]:
# вопрос 4.8

**Факт:** ...

**Предположение:** ...

### 4.9 Что добавляет титул из имени (Title)?

In [14]:
# вопрос 4.9

**Факт:** ...

**Предположение:** ...

### 4.10 Какие новые признаки оказались наиболее информативными?

In [15]:
# вопрос 4.10

**Факт:** ...

**Предположение:** ...

## 5. NumPy-часть

Массив из числовых признаков, ручная стандартизация, матрица корреляций, векторизованная доля выживших по группам, сравнение скорости с циклом, ручная проверка на маленьком примере.

In [ ]:
# блок 5


### 1.5 Гипотезы до очистки

Записаны после первичного аудита, но **до** очистки и до расчёта связей признаков с `Survived`. Основание — результаты аудита и общеизвестная история крушения, а не данные этого набора. Формулировки говорят о **связи**, а не о причине. В разделе 4 к каждой вернёмся и отметим: подтвердилась / частично / не подтвердилась.

**О выживаемости**

| № | Гипотеза | Основание | Где проверим | Что её опровергнет |
|---|---|---|---|---|
| H1 | Женщины выживали заметно чаще мужчин | правило «женщины и дети первыми» | 4.2 | разница долей меньше 10 п. п. |
| H2 | Чем выше класс билета, тем выше доля выживших: 1-й > 2-й > 3-й | предположение: пассажирам высших классов было проще добраться до шлюпок | 4.2 | порядок классов нарушен |
| H3 | Класс различает пассажиров внутри каждого пола: среди женщин 3-го класса доля выживших ниже, чем среди женщин 1–2 классов | эффекты пола и класса могут не складываться | 4.3 | у женщин всех классов доли почти равны |
| H4 | Дети младше 12 лет выживали чаще взрослых | правило «женщины и дети первыми»; в аудите есть 7 младенцев | 4.4 | у детей доля не выше, чем у взрослых |
| H5 | Связь с размером семьи нелинейна: в семьях из 2–4 человек доля выживших выше, чем у одиночек и у семей из 5 и более человек | предположение: небольшой группе проще держаться вместе, очень большой — сложнее | 4.5 | доля не зависит от размера семьи или меняется монотонно |
| H6 | Одиночки выживали реже пассажиров с семьёй, но часть разницы объясняется составом группы (среди одиночек больше мужчин): внутри каждого пола разница меньше | предположение: `IsAlone` частично повторяет пол | 4.6 | внутри пола разница не меньше, чем в целом |
| H7 | Различия между портами в основном объясняются составом по классам: внутри каждого класса разница между портами мала | предположение: на разных портах садились пассажиры разных классов | 4.7 | разница между портами сохраняется внутри каждого класса |
| H8 | Стоимость билета и известная каюта (палуба) положительно связаны с выживанием, но в основном через класс: внутри класса связь слабее | `Fare` и `Cabin` тесно связаны с классом (см. D2, D3) | 4.8 | внутри класса связь так же сильна, как в целом |
| H9 | Среди новых признаков самым информативным окажется `Title` | титул объединяет пол, возраст и семейное положение | 4.9–4.10 | другой признак даёт больший разброс долей выживших между группами |

**О качестве данных** (влияют на очистку, проверим в разделе 2)

| № | Гипотеза | Что это значит для очистки | Что её опровергнет |
|---|---|---|---|
| D1 | Пропуски `Age` неслучайны: в 3-м классе их больше, чем в 1-м и 2-м | не заполнять одной общей медианой, а брать медиану по группам (`Title`, `Pclass`) и добавить признак «возраст был пропущен» | доля пропусков в классах почти одинакова |
| D2 | Пропуск `Cabin` несёт информацию: каюта известна в основном у пассажиров 1-го класса | не заполнять; `Deck = "Unknown"` как отдельная категория и признак `HasCabin` | доля пропусков в классах почти одинакова |
| D3 | `Fare` — цена билета целиком, а не одного человека: на общем билете она одинакова, а в пересчёте на человека близка к обычной для класса | ввести `FarePerPerson = Fare / TicketGroupSize` | цена на человека остаётся сильно выше обычной для класса |
| D4 | `FamilySize` (по `SibSp` и `Parch`) и `TicketGroupSize` (по билету) совпадают не всегда: часть семей разделена между `train.csv` и `test.csv`, а на одном билете бывают не только родственники | оба признака сохранить, расхождения разобрать в проверке согласованности | признаки совпадают почти везде |


### 5.1 Вывод

...

## 6. Итоговая визуализация и таблица результатов

In [17]:
# блок 6

## 7. Выводы

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...
7. ...
8. ...
9. ...
10. ...

### Ограничения анализа

- ...

### Как продолжить исследование

1. ...
2. ...
3. ...

## 8. Дополнительно: rule-based прогноз

Простые правила без sklearn, accuracy против наивного прогноза «все погибли».

In [18]:
# блок 8

## Перед сдачей

- [ ] `Restart Kernel and Run All` проходит без ошибок
- [ ] все пути относительные, seed зафиксирован
- [ ] нет неиспользуемых импортов, экспериментов и закомментированного кода
- [ ] у каждого вопроса: расчёт → график → вывод (факт / предположение)
- [ ] в репозитории: notebook, README, `data/processed/titanic_clean.csv`, таблица результатов, итоговый PNG, `requirements.txt`